<a href="https://colab.research.google.com/github/faizanxmd/1ms25is068_assignment2/blob/main/agent_loop_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Agent Loop Demo


This notebook demonstrates the core mechanism behind an AI agent: a model that can decide to call a tool, receive the result, and continue reasoning toward a goal — all in a loop.

---

This notebook contains starter code. Please fill in details as needed.
---

## Setup - Gemini API key

How to get the key (takes about two minutes):

* Go to [aistudio.google.com](https://aistudio.google.com/prompts/new_chat)
* Sign in with your Google account
* Click "Get API key" in the left sidebar
* Click "Create API key" — choose "Create API key in new project" if prompted
Copy the key that appears (it starts with AIza...)

> The free tier allows up to 10 requests per minute and 250 requests per day, which is enough to work through this notebook.

If you see occasional slow-downs when you run several questions back to back — that is normal, and the code handles it automatically.

## References
* [Google agent loop](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)
* https://ai.google.dev/gemini-api/docs/thinking#signatures
* https://ai.google.dev/gemini-api/docs/text-generation$0
* [Developer guide](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)



In [ ]:
import os

# NEVER share API keys directly. Save them in the environment.
try:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    print('API key loaded from Colab Secrets.')

# Fall back to manual entry if Secrets are not set up.
# REMOVE API keys before pushing code to Github. This fallback helps.
except Exception:
    import getpass
    key = getpass.getpass('Paste your Gemini API key and press Enter: ')
    os.environ['GOOGLE_API_KEY'] = key
    print('API key set.')

In [ ]:
# Quiet ensures your UI is not full of logs.
!pip install google-generativeai google-genai --quiet
print("Done.")

In [ ]:
# @title User input needed.

# Documents about cricket that the agent can search.
DOCUMENTS = {
    "cricket_basics": (
        "Cricket is a bat-and-ball game played between two teams of eleven players. "
        "It is played on a field with a 22-yard pitch at the centre, with a wicket at each end. "
        "The batting side scores runs by running between the wickets or hitting the ball to the boundary. "
        "A ball that crosses the boundary after bouncing scores four runs, and one that clears it on the full scores six runs. "
        "The fielding side tries to get batters out, for example by bowling them, catching the ball or running them out."
    ),
    "formats": (
        "International cricket is played in three main formats: Test, ODI and T20. "
        "A Test match lasts up to five days and each team bats twice. "
        "A One Day International (ODI) gives each team 50 overs, and a T20 match gives each team 20 overs. "
        "An over is a set of six legal deliveries bowled by one bowler. "
        "T20 matches usually finish in about three and a half hours, which makes them popular with television audiences."
    ),
    "ipl": (
        "The Indian Premier League (IPL) is a professional T20 league that started in 2008. "
        "Teams represent Indian cities, such as Mumbai Indians, Chennai Super Kings and Royal Challengers Bengaluru. "
        "Players are bought by franchises in an auction held before the season. "
        "Each team may include a limited number of overseas players in its playing eleven. "
        "The impact player rule lets a team substitute one player during a match."
    ),
    "equipment": (
        "A cricket ball is made of cork wrapped in string and covered with leather. "
        "Test matches traditionally use a red ball, while white balls are used in ODIs and T20s, and pink balls in day-night Tests. "
        "The bat is made of willow and may be at most 108 mm wide. "
        "Batters wear pads, gloves and a helmet for protection against fast bowling. "
        "The wicket consists of three stumps topped by two bails."
    ),
    "drs": (
        "The Decision Review System (DRS) lets players challenge an umpire's decision using technology. "
        "Ball-tracking predicts whether a delivery would have hit the stumps for leg before wicket (LBW) decisions. "
        "UltraEdge detects faint sounds to show whether the ball touched the bat. "
        "Each team gets a limited number of unsuccessful reviews per innings. "
        "If ball-tracking shows the ball only clipping the stumps, the result is called umpire's call and the original decision stands."
    ),
}

In [ ]:
# @title Define a tool
def search_documents(query: str) -> str:
    """Search the document collection for paragraphs relevant to the query."""
    query_lower = query.lower()
    results = []

    for doc_name, content in DOCUMENTS.items():
        words = set(query_lower.split())
        match_count = sum(1 for word in words if word in content.lower())
        if match_count > 0:
            results.append((match_count, doc_name, content))

    results.sort(reverse=True)

    if not results:
        return "No relevant documents found for that query."

    # Return the top two matches
    output = []
    for _, name, content in results[:2]:
        output.append(f"[Source: {name}]\n{content}")
    return "\n\n".join(output)


# Sanity check for regex match.
print(search_documents("how many overs in a T20 match"))


In [ ]:
# @title Ask gemini to search for something
import google.generativeai as genai
import time

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

model = genai.GenerativeModel(
    # Model name taken from the Gemini developer docs / sample colab.
    model_name="gemini-3.5-flash-lite",
    tools=[search_documents],
    system_instruction=(
        '''
        You are a cricket analyst who answers questions about cricket rules, formats and leagues.
        Think step by step: decide what information you need, then use the
        search_documents tool to look it up before answering.
        Only answer based on what you find in the documents.
        If the documents do not contain the answer, say so clearly.
        '''
    )
)

print("Model ready:", model.model_name)

In [ ]:
import time
from google import genai
from google.genai import types

client = genai.Client()  # reads GEMINI_API_KEY / GOOGLE_API_KEY from env

search_tool = types.Tool(function_declarations=[
    types.FunctionDeclaration(
        name="search_documents",
        description="Search the documents to find most the relevant paragraph.",
        parameters={
            "type": "object",
            "properties": {"query": {"type": "string"}},
            "required": ["query"],
        },
    )
])

config = types.GenerateContentConfig(
    tools=[search_tool],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
)


def send_with_retry(contents, max_retries=3):
    for attempt in range(max_retries):
        try:
            return client.models.generate_content(
                model=model.model_name,
                contents=contents, config=config
            )
        except Exception as e:
            if "429" in str(e) and attempt < max_retries - 1:
                wait = 20 * (attempt + 1)
                print(f"  Rate limit hit, waiting {wait}s...")
                time.sleep(wait)
            else:
                raise


def run_agent(question: str, max_steps: int = 8) -> str:
    print(f"Question: {question}\n" + "-" * 50)
    contents = [types.Content(role="user", parts=[types.Part(text=question)])]

    for step in range(1, max_steps + 1):
        response = send_with_retry(contents)
        model_content = response.candidates[0].content

        # Append the model's turn UNMODIFIED, so thought signatures are preserved
        contents.append(model_content)

        calls = [p.function_call for p in model_content.parts if p.function_call]
        if not calls:
            print(f"\nFinal answer:\n{response.text}")
            return response.text

        # Run every requested tool call, reply with all results in ONE user turn
        result_parts = []
        for call in calls:
            args = dict(call.args)
            print(f"Step {step}: model calls {call.name}({args})")
            result = search_documents(**args)
            print(f"  Result preview: {str(result)[:100]}...")
            result_parts.append(
                types.Part.from_function_response(
                    name=call.name, response={"result": result}
                )
            )
        contents.append(types.Content(role="user", parts=result_parts))

    return "(Stopped: hit max_steps)"


_ = run_agent("How many overs does each team get in an ODI?")

In [ ]:
# @title Ask follow up questions

questions = [
    "What colour ball is used in a day-night Test match?",
    "Which franchises are mentioned as IPL teams, and when did the IPL start?",
    "What happens in DRS when ball-tracking shows umpire's call?",
    # This question is not answered by the database.
    "How many ghosts eat carrots everyday?",
    # This question is part of the general knowledge of the model.
    "What is the most popular food item in Canada?"
]

for i, q in enumerate(questions):
    if i > 0:
        time.sleep(15)  # stay within the free-tier rate limit.
    print("\n" + "=" * 60)
    run_agent(q)


In [ ]:
# This cell will check if you have added atleast 2 more questions.
assert(len(questions) >= 4)
# This will check if you added a question.
assert("Add a question here." not in questions)